# Hospital Data Cleaning - Milestone 1

This notebook documents the cleaning and transformation workflow for the MedTrack_DV hospital operations dataset.

## Cleaning Goals

- Remove duplicate admission records.
- Handle missing patient and operational values.
- Standardize department names.
- Create Tableau-ready date and utilization fields.
- Confirm dataset completeness is above 95%.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
RAW_PATH = ROOT / 'data' / 'raw' / 'hospital_raw_data.csv'
CLEAN_PATH = ROOT / 'data' / 'processed' / 'hospital_cleaned.csv'

raw = pd.read_csv(RAW_PATH)
raw.head()

In [ ]:
raw.shape, raw.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
DEPARTMENT_MAP = {
    'general medicine': 'General Medicine',
    'gen. medicine': 'General Medicine',
    'surgery': 'Surgery',
    'surgical': 'Surgery',
    'orthopedics': 'Orthopedics',
    'orthopaedics': 'Orthopedics',
    'cardiology': 'Cardiology',
    'pediatrics': 'Pediatrics',
    'paediatrics': 'Pediatrics',
    'emergency': 'Emergency',
    'er': 'Emergency',
    'icu': 'ICU',
    'intensive care': 'ICU',
}

cleaned = raw.drop_duplicates(subset=['admission_id'], keep='first').copy()

department_values = cleaned['department'].fillna('').astype(str).str.strip().str.lower()
mode_department = department_values[department_values != ''].mode().iat[0]
cleaned['department'] = department_values.replace({'': mode_department}).map(lambda value: DEPARTMENT_MAP.get(value, value.title()))
cleaned['insurance_type'] = cleaned['insurance_type'].fillna('Unknown').replace('', 'Unknown')
cleaned['gender'] = cleaned['gender'].fillna('Unknown').replace('', 'Unknown')
cleaned['age'] = pd.to_numeric(cleaned['age'], errors='coerce')
cleaned['age'] = cleaned['age'].fillna(cleaned['age'].median()).astype(int)

cleaned['admission_date'] = pd.to_datetime(cleaned['admission_date'])
cleaned['discharge_date'] = pd.to_datetime(cleaned['discharge_date'])
cleaned['length_of_stay_days'] = (cleaned['discharge_date'] - cleaned['admission_date']).dt.days.clip(lower=1)
cleaned['admission_month'] = cleaned['admission_date'].dt.to_period('M').astype(str)
cleaned['admission_year'] = cleaned['admission_date'].dt.year
cleaned['admission_quarter'] = 'Q' + cleaned['admission_date'].dt.quarter.astype(str)

for column in ['total_beds', 'department_bed_capacity', 'occupied_beds', 'staff_available', 'staff_required', 'equipment_available', 'equipment_in_use', 'readmission_flag']:
    cleaned[column] = pd.to_numeric(cleaned[column], errors='coerce')

cleaned['bed_utilization_rate'] = (cleaned['occupied_beds'] / cleaned['department_bed_capacity'] * 100).round(2)
cleaned['equipment_utilization_rate'] = (cleaned['equipment_in_use'] / cleaned['equipment_available'] * 100).round(2)
cleaned['staff_coverage_rate'] = (cleaned['staff_available'] / cleaned['staff_required'] * 100).round(2)
cleaned['is_readmission'] = cleaned['readmission_flag'].astype(int)

cleaned = cleaned.sort_values(['admission_date', 'hospital_id', 'department', 'admission_id']).reset_index(drop=True)
cleaned['admission_date'] = cleaned['admission_date'].dt.date.astype(str)
cleaned['discharge_date'] = cleaned['discharge_date'].dt.date.astype(str)

cleaned.to_csv(CLEAN_PATH, index=False)
cleaned.head()

In [ ]:
total_cells = cleaned.shape[0] * cleaned.shape[1]
missing_cells = cleaned.isna().sum().sum()
completeness = (1 - missing_cells / total_cells) * 100

{
    'raw_rows': len(raw),
    'cleaned_rows': len(cleaned),
    'duplicates_removed': len(raw) - len(cleaned),
    'missing_cells_after_cleaning': int(missing_cells),
    'completeness_percent': round(completeness, 2),
    'departments': sorted(cleaned['department'].unique().tolist()),
}